# 01 - Regularization and Complexity Tuning
Ajuste de regularizacion y complejidad para modelos de retornos y clasificacion.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1",
    "precio_provincial_lag_2",
    "precio_provincial_lag_3",
    "precio_vecinos_media_lag1",
    "precio_nacional_base_ma3",
    "precio_nacional_base_ma6",
    "precio_nacional_base_vol3",
    "precio_nacional_base_vol6"
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for targets")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    y_train_clf = (y_train_reg - base_train > 0).astype(int)
    y_test_clf = (y_test_reg - base_test > 0).astype(int)
    return y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test

def regression_metrics(y_true, y_pred):
    aligned = pd.concat([y_true, y_pred], axis=1).dropna()
    if aligned.empty:
        return {"MAE": np.nan, "RMSE": np.nan, "Pearson": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    y_pred_clean = aligned.iloc[:, 1]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    return {"MAE": float(mae), "RMSE": float(rmse), "Pearson": float(pearson) if pearson == pearson else np.nan}

def classification_metrics(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    return {"DA": float(acc)}

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


## 1. Baseline V1 (retornos + clasificacion calibrada)
Reentrenar baseline V1 para comparar con periodo crisis 2022H1.

In [3]:
tscv = TimeSeriesSplit(n_splits=5)

baseline = {}

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    reg_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ])
    reg_model.fit(X_train_h, y_train_ret)
    pred_ret = pd.Series(reg_model.predict(X_test_h), index=y_test_ret.index)
    reg_metrics = regression_metrics(y_test_ret, pred_ret)

    train_mask_c = y_train_clf.notna()
    test_mask_c = y_test_clf.notna()
    X_train_c = X_train.loc[train_mask_c]
    X_test_c = X_test.loc[test_mask_c]
    y_train_c = y_train_clf.loc[train_mask_c]
    y_test_c = y_test_clf.loc[test_mask_c]

    clf_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ])
    clf_model.fit(X_train_c, y_train_c)
    calib = CalibratedClassifierCV(estimator=clf_model, method="sigmoid", cv=TimeSeriesSplit(n_splits=3))
    calib.fit(X_train_c, y_train_c)
    proba_cal = calib.predict_proba(X_test_c)[:, 1]
    pred_cal = (proba_cal >= 0.5).astype(int)
    clf_metrics = classification_metrics(y_test_c, proba_cal, pred_cal)

    baseline[h] = {
        "reg_model": reg_model,
        "reg_metrics": reg_metrics,
        "y_test_ret": y_test_ret,
        "X_test": X_test_h,
        "clf_model": calib,
        "clf_metrics": clf_metrics,
        "y_test_clf": y_test_c,
        "X_test_clf": X_test_c,
    }

print({h: baseline[h]["reg_metrics"] for h in horizons})

{1: {'MAE': 0.0765821459697079, 'RMSE': 0.0991756265306218, 'Pearson': 0.3471010029150682}, 2: {'MAE': 0.10514975132029854, 'RMSE': 0.1354366633648677, 'Pearson': 0.40677949177720457}, 3: {'MAE': 0.11909221588120229, 'RMSE': 0.15238238878830576, 'Pearson': 0.4544569977722088}}


## 2. Ridge con L2 amplio
GridSearch sobre alpha (1e-3 a 1e5) para retornos.

In [4]:
ridge_results = []
ridge_models = {}

alphas = np.logspace(-3, 5, 9)

for h in horizons:
    y_train_reg, y_test_reg, _, _, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    ridge = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ])
    grid = GridSearchCV(ridge, {"model__alpha": alphas}, cv=tscv, scoring="neg_mean_absolute_error")
    grid.fit(X_train_h, y_train_ret)
    best = grid.best_estimator_
    pred = pd.Series(best.predict(X_test_h), index=y_test_ret.index)
    metrics = regression_metrics(y_test_ret, pred)
    ridge_models[h] = {"model": best, "metrics": metrics, "best_alpha": grid.best_params_["model__alpha"]}
    ridge_results.append({"horizon": h, "best_alpha": grid.best_params_["model__alpha"], **metrics})

ridge_results_df = pd.DataFrame(ridge_results)
ridge_results_df

,horizon,best_alpha,MAE,RMSE,Pearson
0,1,100000.0,0.051800,0.072070,0.292879
1,2,100000.0,0.068548,0.098267,0.331137
2,3,100000.0,0.085135,0.122098,0.384287


## 3. Regularizacion XGBoost (L1/L2) y profundidad
RandomizedSearch sobre reg_alpha, reg_lambda y max_depth.

In [5]:
xgb_results = []
xgb_models = {}

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [2, 3, 4],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.9, 1.0],
    "model__reg_lambda": [1, 3, 10, 30],
    "model__reg_alpha": [0, 0.5, 1.0, 3.0],
}

for h in horizons:
    y_train_reg, y_test_reg, _, _, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    xgb = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror")),
    ])
    search = RandomizedSearchCV(xgb, param_grid, n_iter=20, cv=tscv, scoring="neg_mean_absolute_error", random_state=42, n_jobs=-1)
    search.fit(X_train_h, y_train_ret)
    best = search.best_estimator_
    pred = pd.Series(best.predict(X_test_h), index=y_test_ret.index)
    metrics = regression_metrics(y_test_ret, pred)
    xgb_models[h] = {"model": best, "metrics": metrics, "best_params": search.best_params_}
    xgb_results.append({"horizon": h, **search.best_params_, **metrics})

xgb_results_df = pd.DataFrame(xgb_results)
xgb_results_df

,horizon,model__subsample,model__reg_lambda,model__reg_alpha,model__n_estimators,model__max_depth,model__learning_rate,model__colsample_bytree,MAE,RMSE,Pearson
0,1,1.0,10,3.0,400,4,0.10,0.7,0.052144,0.070027,0.290378
1,2,0.7,3,0.5,400,2,0.05,0.9,0.065615,0.095641,0.535772
2,3,0.7,3,0.5,400,2,0.05,0.9,0.092727,0.132698,0.286620


## 4. Reduccion de profundidad RF/XGB
Compara max_depth mas bajo en retorno y clasificacion.

In [6]:
depth_results = []

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf, base_train, base_test = build_targets(h)
    train_mask_h = base_train.notna() & (base_train != 0) & y_train_reg.notna()
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    base_train_h = base_train.loc[train_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    y_train_ret = (y_train_reg.loc[train_mask_h] - base_train_h) / base_train_h
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    for depth in [2, 3, 4]:
        rf = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1, max_depth=depth)),
        ])
        rf.fit(X_train_h, y_train_ret)
        pred = pd.Series(rf.predict(X_test_h), index=y_test_ret.index)
        metrics = regression_metrics(y_test_ret, pred)
        depth_results.append({"horizon": h, "model": "RF", "max_depth": depth, **metrics})

        xgb = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(random_state=42, n_jobs=-1, max_depth=depth, objective="reg:squarederror")),
        ])
        xgb.fit(X_train_h, y_train_ret)
        pred_x = pd.Series(xgb.predict(X_test_h), index=y_test_ret.index)
        metrics_x = regression_metrics(y_test_ret, pred_x)
        depth_results.append({"horizon": h, "model": "XGB", "max_depth": depth, **metrics_x})

depth_results_df = pd.DataFrame(depth_results)
depth_results_df

,horizon,model,max_depth,MAE,RMSE,Pearson
0,1,RF,2,0.060391,0.083517,0.245553
1,1,XGB,2,0.049630,0.067789,0.373248
2,1,RF,3,0.069721,0.094177,0.294994
3,1,XGB,3,0.051003,0.072438,0.368922
4,1,RF,4,0.075748,0.098705,0.338910
5,1,XGB,4,0.062790,0.082823,0.012962
6,2,RF,2,0.093199,0.127366,0.229789
7,2,XGB,2,0.069434,0.101986,0.476625
8,2,RF,3,0.098695,0.132316,0.327782
9,2,XGB,3,0.077100,0.108029,0.333130


## 5. Evaluacion anti-crisis (2022-01 a 2022-06)
Comparar MAE en periodo critico vs baseline V1.

In [8]:
crisis_mask = (test_df["date"] >= pd.Timestamp("2022-01-01")) & (test_df["date"] <= pd.Timestamp("2022-06-30"))

anti_rows = []

for h in horizons:
    y_train_reg, y_test_reg, _, _, base_train, base_test = build_targets(h)
    test_mask_h = base_test.notna() & (base_test != 0) & y_test_reg.notna()
    X_test_h = X_test.loc[test_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    y_test_ret = (y_test_reg.loc[test_mask_h] - base_test_h) / base_test_h

    crisis_idx = y_test_ret.index.intersection(test_df.loc[crisis_mask].index)

    base_pred = pd.Series(baseline[h]["reg_model"].predict(X_test_h), index=y_test_ret.index)
    base_mae = mean_absolute_error(y_test_ret.loc[crisis_idx], base_pred.loc[crisis_idx])

    ridge_pred = pd.Series(ridge_models[h]["model"].predict(X_test_h), index=y_test_ret.index)
    ridge_mae = mean_absolute_error(y_test_ret.loc[crisis_idx], ridge_pred.loc[crisis_idx])

    xgb_pred = pd.Series(xgb_models[h]["model"].predict(X_test_h), index=y_test_ret.index)
    xgb_mae = mean_absolute_error(y_test_ret.loc[crisis_idx], xgb_pred.loc[crisis_idx])

    anti_rows.append({
        "horizon": h,
        "baseline_mae": base_mae,
        "ridge_mae": ridge_mae,
        "xgb_mae": xgb_mae,
        "ridge_improve": (base_mae - ridge_mae) / base_mae if base_mae else np.nan,
        "xgb_improve": (base_mae - xgb_mae) / base_mae if base_mae else np.nan,
    })

anti_df = pd.DataFrame(anti_rows)
anti_df

report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "OPTIMIZACION_REGULARIZACION_V1.md"

lines = [
    "# OPTIMIZACION_REGULARIZACION_V1",
    "",
    "## Resumen",
    "Busqueda de regularizacion y complejidad para mejorar estabilidad en 2022H1.",
    "",
    "## Ridge (mejor alpha por horizonte)",
    ridge_results_df.to_markdown(index=False),
    "",
    "## XGBoost (mejores params)",
    xgb_results_df.to_markdown(index=False),
    "",
    "## Complejidad (max_depth bajo)",
    depth_results_df.to_markdown(index=False),
    "",
    "## Anti-crisis 2022H1 (MAE vs baseline)",
    anti_df.to_markdown(index=False),
    "",
]

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\reports\OPTIMIZACION_REGULARIZACION_V1.md
